# Modelo Scoring de Riesgo — Nave

**Autor:** data-team  
**Última ejecución:** manual  
**Frecuencia:** semanal (lunes a la mañana)  

Este notebook genera el score de riesgo de los clientes activos.
El resultado se guarda en `/data/output/scoring_semanal.csv` y el equipo de riesgo lo levanta desde ahí.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report
import pickle
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Carga de datos
# NOTA: el CSV viene del proceso de extracción que corre Infraestructura los domingos
# Si no está el archivo, hay que pedirle a Martín que corra el job de extracción
import os
if os.path.exists('/data/clientes_activos.csv'):
    df = pd.read_csv('/data/clientes_activos.csv')
else:
    # DEMO: datos sintéticos que replican la estructura real
    np.random.seed(42)
    n = 5000
    df = pd.DataFrame({
        'cliente_id': range(1, n + 1),
        'edad': np.random.randint(18, 75, n),
        'antiguedad_meses': np.random.randint(1, 240, n),
        'saldo_promedio_90d': np.random.exponential(50000, n),
        'cant_productos': np.random.randint(1, 8, n),
        'dias_ultimo_movimiento': np.random.randint(0, 365, n),
        'ratio_utilizacion_credito': np.random.uniform(0, 1, n),
        'cant_cuotas_atrasadas': np.random.poisson(0.3, n),
        'segmento': np.random.choice(['RETAIL', 'PYME', 'CORPORATIVO'], n, p=[0.7, 0.2, 0.1]),
        'default_90d': np.random.binomial(1, 0.08, n),
    })
    print('[DEMO] Usando datos sintéticos — estructura idéntica al dataset real de Nave')

print(f'Registros cargados: {len(df)}')
df.head()

In [ ]:
# Preprocesamiento
# Imputar nulos con mediana (acordado con el equipo de riesgo en julio 2024)
features = ['edad', 'antiguedad_meses', 'saldo_promedio_90d', 
            'cant_productos', 'dias_ultimo_movimiento', 
            'ratio_utilizacion_credito', 'cant_cuotas_atrasadas']

df[features] = df[features].fillna(df[features].median())

# Encoding de variables categóricas
df['segmento_encoded'] = df['segmento'].map({'RETAIL': 0, 'PYME': 1, 'CORPORATIVO': 2})
df['segmento_encoded'] = df['segmento_encoded'].fillna(0)  # los nuevos van a RETAIL

all_features = features + ['segmento_encoded']
X = df[all_features]
y = df['default_90d']  # 1 = entró en mora en los últimos 90 días

In [ ]:
# Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
# Entrenamiento
# Hiperparámetros: los que dieron mejor resultado en la exploración de diciembre 2023
# No tocar sin hablar con el equipo de riesgo
model = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    random_state=42
)
model.fit(X_train, y_train)

In [ ]:
# Evaluación
y_pred_proba = model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_pred_proba)
print(f'AUC-ROC: {auc:.4f}')
print(classification_report(y_test, model.predict(X_test)))

In [ ]:
# Scoring de todos los clientes activos
X_all_scaled = scaler.transform(df[all_features])
df['score_riesgo'] = model.predict_proba(X_all_scaled)[:, 1]
df['riesgo_alto'] = (df['score_riesgo'] >= 0.6).astype(int)

print(f"Clientes con riesgo alto: {df['riesgo_alto'].sum()} ({df['riesgo_alto'].mean():.1%})")

In [ ]:
# Guardar resultados
import os
os.makedirs('/tmp/output', exist_ok=True)
os.makedirs('/tmp/models', exist_ok=True)

output_cols = ['cliente_id', 'score_riesgo', 'riesgo_alto']
df[output_cols].to_csv('/tmp/output/scoring_semanal.csv', index=False)
print('Archivo guardado en /tmp/output/scoring_semanal.csv')

# Guardar modelo (por si hay que reproducir)
with open('/tmp/models/modelo_riesgo_v1.pkl', 'wb') as f:
    pickle.dump({'model': model, 'scaler': scaler}, f)
print('Modelo guardado.')
print(f'\nPreview de resultados:')
print(df[output_cols].head())